# Shift-Scan — Controlled Validation Suite
Fills the gaps a professor will probe: **fair decay-only baselines, real decay traces, long-context loss-by-position, per-layer sensitivity, and an honest hardware analysis.**

| # | Experiment | Needs training? | Closes gap |
|---|---|---|---|
| 1 | Retention-product error on **real** Mamba-2 decay traces | No | HARPQ motivation (replaces synthetic traces) |
| 2 | Decay-only PTQ: FP vs Nearest-PoT vs HARPQ vs Dyadic2, paired bootstrap CIs | No | Phase A (fair baselines) |
| 3 | Loss-by-position up to 4096 tokens | No | Long-context stability |
| 4 | Per-layer sensitivity (decay + ternary projections) | No | Phase C / which layers stay FP |
| 5 | Memory-bound roofline + GEMV timing | No | Why the Triton kernel was slow |

**Honesty notes:** HARPQ / Dyadic2 below are *my reconstructions* from your summary (HARPQ = horizon-clipped timescale matching). Swap in your own functions at the marked cell. All numbers come from running this notebook, none are pre-filled.

In [ ]:
!pip -q install -U transformers datasets accelerate scipy pandas matplotlib

In [ ]:
import torch, torch.nn.functional as F, numpy as np, pandas as pd, matplotlib.pyplot as plt
import json, os, time, zipfile
from transformers import AutoTokenizer, Mamba2ForCausalLM
from datasets import load_dataset

MODEL  = "AntonV/mamba2-130m-hf"   # HF-format Mamba-2 130M; change if you use another checkpoint
dev    = "cuda" if torch.cuda.is_available() else "cpu"
OUT    = "/content/shiftscan_validation"; os.makedirs(OUT, exist_ok=True)
SEED   = 0; torch.manual_seed(SEED)
RES    = {}   # all results go here -> results.json

In [ ]:
def load_model():
    tok = AutoTokenizer.from_pretrained(MODEL)
    m = Mamba2ForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32).to(dev).eval()
    return tok, m

def load_tokens(tok, n_tokens, path="/content/eval_tokens.pt"):
    if os.path.exists(path): return torch.load(path)          # fixed eval stream => reproducible
    ds = load_dataset("HuggingFaceFW/fineweb-edu", "sample-10BT", split="train", streaming=True)
    ids = []
    for ex in ds:
        ids += tok(ex["text"])["input_ids"] + [tok.eos_token_id]
        if len(ids) >= n_tokens: break
    t = torch.tensor(ids[:n_tokens]); torch.save(t, path); return t

tok, model = load_model()
TOK = load_tokens(tok, 300_000)
mixers = [l.mixer for l in model.backbone.layers]; NL = len(mixers)
print(f"{NL} layers | heads={mixers[0].num_heads} | eval tokens={len(TOK):,} | device={dev}")

## Decay hook
Intercepts the `dt` channel of each mixer's `in_proj` to (a) **capture** the true per-token decay `a = exp(-dt·A)` and (b) **replace** it with a quantized `â` by inverting softplus. No kernel changes, so it works with both fast and slow Mamba-2 paths.

In [ ]:
CAP, QFN, QLAYERS = None, None, set()

def make_hook(m, i):
    def hook(mod, inp, out):
        H   = m.num_heads
        dtr = out[..., -H:].float()                                   # split order: gate | conv(x,B,C) | dt
        dt  = F.softplus(dtr + m.dt_bias.float())
        lo, hi = getattr(m, "time_step_limit", (0.0, float("inf")))
        dt  = dt.clamp(lo, hi)
        A   = torch.exp(m.A_log.float())                              # positive rate
        a   = torch.exp(-dt * A)
        if CAP is not None: CAP[i].append(a.detach().cpu())
        if QFN is not None and i in QLAYERS:
            ah  = QFN(a)
            dth = -torch.log(ah) / A
            raw = dth + torch.log(-torch.expm1(-dth)) - m.dt_bias.float()   # inverse softplus
            out = out.clone(); out[..., -H:] = raw.to(out.dtype)
            return out
    return hook

for i, m in enumerate(mixers): m.in_proj.register_forward_hook(make_hook(m, i))

# sanity: with QFN=None the hook must not change outputs
x = TOK[:512].view(1, -1).to(dev)
with torch.no_grad(): l0 = model(input_ids=x, use_cache=False).logits
print("hook installed; logits finite:", torch.isfinite(l0).all().item())

## Decay quantizers  ·  ← replace with your own HARPQ / Dyadic2 if they differ

In [ ]:
KS = torch.arange(1, 17, device=dev).float()
CAND = 1 - 2.0 ** (-KS)                                             # 1 - 2^-k, k = 1..16
S2 = torch.tensor(sorted({2.**-i + 2.**-j for i in range(1,17) for j in range(i,17)} | {2.**-i for i in range(1,17)}))
S2 = S2[S2 < 1].to(dev)                                             # dyadic-2 values of (1 - a)

def q_nearest(a):                                                   # nearest k in log(1-a)
    k = torch.round(-torch.log2((1 - a).clamp_min(2.0**-16))).clamp(1, 16)
    return 1 - 2.0 ** (-k)

def q_harpq(a, H=64, c=4):                                          # horizon-clipped timescale matching (my reconstruction)
    tau  = (-1 / torch.log(a)).clamp(max=c * H)                     # memory timescale, clipped at c*H tokens
    tauc = (-1 / torch.log(CAND)).clamp(max=c * H)
    return CAND[(torch.log(tauc) - torch.log(tau).unsqueeze(-1)).abs().argmin(-1)]
    # NOTE: matching a^H directly is degenerate for fast channels (a^H~0 for every candidate)

def q_dyadic2(a):                                                   # 1-a ≈ 2^-i + 2^-j
    s = (1 - a).clamp_min(1e-6).unsqueeze(-1)
    return 1 - S2[(torch.log(s) - torch.log(S2)).abs().argmin(-1)]

METHODS = {"Nearest PoT": q_nearest, "HARPQ": q_harpq, "Dyadic2": q_dyadic2}

## Experiment 1 — Retention-product error on real decay traces
For window `H`, exact retention is ∏a_t; we measure how far each quantizer drifts. This directly tests the horizon-aware claim on **measured** (not synthetic) Mamba-2 traces.

In [ ]:
def collect_traces(T=2048, n=4):
    global CAP
    CAP = {i: [] for i in range(NL)}
    x = TOK[:n*T].view(n, T).to(dev)
    with torch.no_grad(): model(input_ids=x, use_cache=False)
    tr = {i: torch.cat(v) for i, v in CAP.items()}; CAP = None
    return tr                                                       # {layer: [n, T, heads]}

def retention_err(a, ah, H):
    la = torch.log(a.clamp_min(1e-12)).double().to(dev); lh = torch.log(ah).double().to(dev)
    ca = F.pad(la.cumsum(1), (0,0,1,0)); ch = F.pad(lh.cumsum(1), (0,0,1,0))
    wa, wh = ca[:, H:] - ca[:, :-H], ch[:, H:] - ch[:, :-H]
    return (wh.exp() - wa.exp()).abs().mean().item(), (wh - wa).abs().mean().item()

TR = collect_traces()
rows = []
for name, q in METHODS.items():
    for H in [16, 64, 256, 1024]:
        e = np.array([retention_err(TR[i].to(dev), q(TR[i].to(dev)), H) for i in range(NL)])
        rows.append(dict(method=name, H=H, abs_retention_err=e[:,0].mean(), mean_abs_log_err=e[:,1].mean()))
df1 = pd.DataFrame(rows); RES["exp1_retention"] = df1.to_dict("records")
print(df1.pivot(index="H", columns="method", values="abs_retention_err").round(5))
print(df1.pivot(index="H", columns="method", values="mean_abs_log_err").round(4))

allv = torch.cat([TR[i].flatten() for i in range(NL)])
plt.figure(figsize=(6,3.2)); plt.hist(allv.numpy(), 120, log=True); plt.xlabel("decay a"); plt.title("Real Mamba-2 130M decay distribution")
plt.tight_layout(); plt.savefig(f"{OUT}/fig_decay_hist.png", dpi=160); plt.show()

## Experiment 2 — Decay-only PTQ with paired bootstrap CIs (Phase A)
FP projections everywhere; only the decay changes. Same tokens for every method, so differences are **paired** and need no training seeds.

In [ ]:
BS = 2
@torch.no_grad()
def run(T, n, qfn=None, layers=None):
    global QFN, QLAYERS
    QFN, QLAYERS = qfn, set(range(NL)) if layers is None else set(layers)
    x = TOK[:n*T].view(n, T).to(dev); out = []
    for i in range(0, n, BS):
        xb = x[i:i+BS]; lg = model(input_ids=xb, use_cache=False).logits.float()
        out.append(F.cross_entropy(lg[:, :-1].reshape(-1, lg.size(-1)), xb[:, 1:].reshape(-1), reduction="none").view(len(xb), -1).cpu())
    QFN = None
    return torch.cat(out)                                           # [n, T-1] per-token CE

def boot(d, B=4000):
    rng = np.random.default_rng(0); d = np.asarray(d); idx = rng.integers(0, len(d), (B, len(d)))
    m = d[idx].mean(1); return d.mean(), np.percentile(m, 2.5), np.percentile(m, 97.5)

T2, N2 = 2048, 64
L = {"FP (baseline)": run(T2, N2)}
for k, q in METHODS.items(): L[k] = run(T2, N2, q)
cl = {k: v.mean(1).numpy() for k, v in L.items()}
rows = []
for k in L:
    d, lo, hi = boot(cl[k] - cl["FP (baseline)"])
    rows.append(dict(method=k, PPL=float(np.exp(cl[k].mean())), dCE_vs_FP=d, CI_lo=lo, CI_hi=hi))
df2 = pd.DataFrame(rows); RES["exp2_decay_only"] = df2.to_dict("records"); print(df2.round(5).to_string(index=False))

def paired(a, b):
    d, lo, hi = boot(cl[a] - cl[b]); sig = "SIGNIFICANT" if (lo > 0 or hi < 0) else "no significant difference"
    print(f"{a} - {b}: dCE={d:+.5f} nats  95% CI [{lo:+.5f}, {hi:+.5f}]  -> {sig}")
    return dict(a=a, b=b, d=d, lo=lo, hi=hi)
RES["exp2_paired"] = [paired("HARPQ","Nearest PoT"), paired("HARPQ","Dyadic2"), paired("Dyadic2","Nearest PoT")]

## Experiment 3 — Long-context loss-by-position (up to 4096 tokens)
If quantized decay accumulates error, loss gap vs FP should **grow with position**. A flat gap is evidence of long-context stability.

In [ ]:
T3, N3 = 4096, 12
LP = {"FP (baseline)": run(T3, N3)}
for k, q in METHODS.items(): LP[k] = run(T3, N3, q)

def smooth(v, w=128): return np.convolve(v, np.ones(w)/w, mode="valid")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
base = LP["FP (baseline)"].mean(0).numpy()
for k, v in LP.items():
    c = v.mean(0).numpy(); ax[0].plot(smooth(c), label=k)
    if k != "FP (baseline)": ax[1].plot(smooth(c - base), label=k)
ax[0].set(title="Loss by position", xlabel="token position", ylabel="CE (nats)"); ax[0].legend()
ax[1].axhline(0, c="k", lw=.5); ax[1].set(title="Gap vs FP by position", xlabel="token position", ylabel="Δ CE"); ax[1].legend()
plt.tight_layout(); plt.savefig(f"{OUT}/fig_loss_by_position.png", dpi=160); plt.show()

rows = []
for k, v in LP.items():
    c = v.mean(0).numpy(); rows.append(dict(method=k, early_0_512=c[:512].mean(), late_3584_4095=c[3584:].mean(), late_minus_early=c[3584:].mean()-c[:512].mean()))
df3 = pd.DataFrame(rows); RES["exp3_long_ctx"] = df3.to_dict("records"); print(df3.round(4).to_string(index=False))

## Experiment 4 — Per-layer sensitivity (Phase C)
One layer at a time: (a) quantize its decay, (b) ternarize its `in_proj`+`out_proj` (TWN, post-training). Large ΔCE ⇒ keep that layer FP. This gives a **data-driven** reason for the FP-sensitive set instead of a hand-picked one.

In [ ]:
def twn(W):
    d = 0.7 * W.abs().mean(1, keepdim=True); m = (W.abs() > d).float()
    al = (W.abs() * m).sum(1, keepdim=True) / m.sum(1, keepdim=True).clamp_min(1)
    return torch.sign(W) * m * al

class Tern:
    def __init__(s, layers): s.layers = layers; s.bak = {}
    def __enter__(s):
        for i in s.layers:
            for n in ("in_proj", "out_proj"):
                p = getattr(mixers[i], n).weight; s.bak[(i, n)] = p.data.clone(); p.data = twn(p.data)
    def __exit__(s, *a):
        for (i, n), w in s.bak.items(): getattr(mixers[i], n).weight.data = w

T4, N4 = 1024, 16
base4 = run(T4, N4).mean().item(); rows = []
for i in range(NL):
    r = dict(layer=i)
    for k in ("Nearest PoT", "HARPQ"): r[f"decay_{k}"] = run(T4, N4, METHODS[k], [i]).mean().item() - base4
    with Tern([i]): r["ternary_proj"] = run(T4, N4).mean().item() - base4
    rows.append(r)
df4 = pd.DataFrame(rows); RES["exp4_layer_sens"] = df4.to_dict("records")
print(df4.round(4).to_string(index=False))
print("Most ternary-sensitive layers (keep FP):", df4.sort_values("ternary_proj", ascending=False).layer.head(6).tolist())

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].bar(df4.layer, df4.ternary_proj); ax[0].set(title="ΔCE: ternarize layer i (PTQ)", xlabel="layer")
w = .4; ax[1].bar(df4.layer-w/2, df4["decay_Nearest PoT"], w, label="Nearest"); ax[1].bar(df4.layer+w/2, df4["decay_HARPQ"], w, label="HARPQ")
ax[1].set(title="ΔCE: quantize decay of layer i", xlabel="layer"); ax[1].legend()
plt.tight_layout(); plt.savefig(f"{OUT}/fig_layer_sensitivity.png", dpi=160); plt.show()

## Experiment 5 — Why the ternary kernel lost: memory-bound roofline
At decode (M=1) a linear layer is **bandwidth-bound**, so the best case for ternary is the byte ratio. This measures FP16 GEMV and what a *perfect* 2-bit kernel could gain. T4 has no ternary hardware, so gains need LUT/bit-trick kernels, not naive masks.

In [ ]:
def bench(fn, it=200):
    for _ in range(20): fn()
    torch.cuda.synchronize(); s = torch.cuda.Event(True); e = torch.cuda.Event(True); s.record()
    for _ in range(it): fn()
    e.record(); torch.cuda.synchronize(); return s.elapsed_time(e) / it

if dev == "cuda":
    W = mixers[0].in_proj.weight.data.half(); N_, K_ = W.shape; rows = []
    for M in (1, 16, 128, 512):
        x = torch.randn(M, K_, device=dev, dtype=torch.half); ms = bench(lambda: x @ W.T)
        by16 = N_ * K_ * 2; by2 = N_ * K_ / 4 + N_ * 2                  # 2-bit packed + fp16 per-row scale
        rows.append(dict(M=M, fp16_ms=ms, eff_GBps=by16/ms/1e6, ideal_2bit_speedup=by16/by2 if M <= 16 else 1.0))
    df5 = pd.DataFrame(rows); RES["exp5_roofline"] = df5.to_dict("records"); print(f"in_proj {N_}x{K_}  (T4 peak ≈ 320 GB/s)"); print(df5.round(3).to_string(index=False))
    print("Ideal speedup applies only when memory-bound (small M). At M=512 it is compute-bound and FP16 tensor cores win.")

## Save everything

In [ ]:
json.dump(RES, open(f"{OUT}/results.json", "w"), indent=1, default=float)
for k, v in RES.items():
    if isinstance(v, list) and v and isinstance(v[0], dict): pd.DataFrame(v).to_csv(f"{OUT}/{k}.csv", index=False)
with zipfile.ZipFile("/content/shiftscan_validation.zip", "w") as z:
    for f in os.listdir(OUT): z.write(f"{OUT}/{f}", f)
try:
    from google.colab import files; files.download("/content/shiftscan_validation.zip")
except Exception: print("saved: /content/shiftscan_validation.zip")

## How to turn the results into resume / paper claims
| If you observe… | Safe claim |
|---|---|
| Exp 2: HARPQ vs Nearest CI excludes 0 (negative) | "HARPQ significantly lowers decay-induced CE (paired bootstrap, 95% CI)" |
| Exp 2: CI includes 0 | "Decay quantization to 2^-k is near-lossless; method choice is not significant" (still a valid finding) |
| Exp 1: HARPQ wins only at H≈64 | "Horizon-aware gain is horizon-specific" — report as a limitation |
| Exp 3: gap flat vs position | "No evidence of error accumulation up to 4096 tokens" |
| Exp 4: few layers dominate | "Keeping k sensitive layers in FP recovers most ternary loss" |
| Exp 5: ideal speedup ≫ measured kernel | "Theoretical savings need bandwidth-aware kernels; naive Triton is 80× slower" |

**Next (needs training):** rerun your v2 best config with `Decay ∈ {None, Nearest, HARPQ, Dyadic2}` × 3 seeds at the same token budget, and add the FP-sensitive layer list from Exp 4.